# 01 · Extract MFCC and HuBERT features

Both feature sets use the same validated English utterances. Each recording is converted to 16 kHz mono, divided into consecutive 10-second chunks, and summarized across all valid frames. Per-utterance caches let this long stage resume.


In [1]:
from pathlib import Path
import json, os, platform, random
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio
from tqdm.auto import tqdm
from transformers import AutoFeatureExtractor, HubertModel

ROOT = Path.cwd()
DATA = ROOT / 'data'
FEATURES = ROOT / 'features'
FEATURES.mkdir(exist_ok=True)
manifest = pd.read_csv(DATA / 'manifest.csv').sort_values('utterance_id').reset_index(drop=True)
assert manifest.utterance_id.is_unique and len(manifest) > 0
SAMPLE_RATE = 16_000
CHUNK_SAMPLES = SAMPLE_RATE * 10
print(len(manifest), 'validated utterances')


C:\Users\renny\Desktop\projects\speech-nli-mfcc-hubert\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


13310 validated utterances


## 1. Shared audio preprocessing

The final short chunk is retained. A remainder under 0.1 second is joined to the prior chunk to avoid an undersized HuBERT input.


In [2]:
def load_audio(path):
    array, rate = sf.read(path, dtype='float32', always_2d=True)
    wave = torch.from_numpy(array.mean(axis=1).copy())
    if rate != SAMPLE_RATE:
        wave = torchaudio.functional.resample(wave, rate, SAMPLE_RATE)
    if not torch.isfinite(wave).all() or wave.numel() < SAMPLE_RATE // 2:
        raise ValueError(f'Invalid waveform: {path}')
    return wave

def audio_chunks(wave):
    pieces = list(wave.split(CHUNK_SAMPLES))
    if len(pieces) > 1 and pieces[-1].numel() < SAMPLE_RATE // 10:
        pieces[-2] = torch.cat((pieces[-2], pieces[-1]))
        pieces.pop()
    return pieces

def summarize_frames(parts):
    frames = torch.cat(parts, dim=0).float()
    return torch.cat((frames.mean(0), frames.std(0, unbiased=False))).numpy().astype('float32')


## 2. MFCC representation

Twenty MFCCs plus their first and second temporal differences produce 60 values per frame. Mean and standard deviation across all frames yield a 120-value utterance vector.


In [3]:
mfcc_transform = torchaudio.transforms.MFCC(
    sample_rate=SAMPLE_RATE, n_mfcc=20, log_mels=True,
    melkwargs={'n_fft':400, 'hop_length':160, 'n_mels':40, 'center':False})

def mfcc_vector(wave):
    parts = []
    for chunk in audio_chunks(wave):
        coefficients = mfcc_transform(chunk)
        delta = torchaudio.functional.compute_deltas(coefficients)
        delta2 = torchaudio.functional.compute_deltas(delta)
        parts.append(torch.cat((coefficients, delta, delta2), dim=0).T)
    return summarize_frames(parts)

def save_vector(cache_dir, utterance_id, vector):
    destination = cache_dir / f'{utterance_id}.npy'
    temporary = destination.with_suffix('.part')
    with temporary.open('wb') as stream:
        np.save(stream, vector)
    os.replace(temporary, destination)


In [4]:
mfcc_cache = FEATURES / 'mfcc_cache'
mfcc_cache.mkdir(exist_ok=True)
for row in tqdm(manifest.itertuples(index=False), total=len(manifest), desc='MFCC'):
    destination = mfcc_cache / f'{row.utterance_id}.npy'
    if not destination.exists():
        save_vector(mfcc_cache, row.utterance_id, mfcc_vector(load_audio(row.path)))
print('MFCC vectors cached:', len(list(mfcc_cache.glob('*.npy'))))


MFCC:   0%|          | 0/13310 [00:00<?, ?it/s]

MFCC:  27%|██▋       | 3571/13310 [00:00<00:00, 35664.80it/s]

MFCC:  56%|█████▌    | 7414/13310 [00:00<00:00, 37069.31it/s]

MFCC:  87%|████████▋ | 11550/13310 [00:00<00:00, 38891.50it/s]

MFCC: 100%|██████████| 13310/13310 [00:00<00:00, 38433.94it/s]

MFCC vectors cached: 13310


## 3. Frozen HuBERT representation on the RTX 4060

The official feature extractor normalizes 16 kHz waveforms. Only the first six transformer layers are retained; their final output is exactly layer 6 of the full model, verified on a sample. Frame mean and standard deviation form a 1,536-value vector. CUDA is required so a resumed cache cannot mix CPU and GPU precision.


In [5]:
assert torch.cuda.is_available(), 'CUDA is required for uniform final HuBERT extraction.'
DEVICE = torch.device('cuda:0')
assert '4060' in torch.cuda.get_device_name(DEVICE)
print('Frozen HuBERT extraction on', torch.cuda.get_device_name(DEVICE))
MODEL_ID = 'facebook/hubert-base-ls960'
MODEL_REVISION = 'dba3bb02fda4248b6e082697eee756de8fe8aa8a'
extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = HubertModel.from_pretrained(MODEL_ID, revision=MODEL_REVISION,
                                   use_safetensors=False).eval()
model.encoder.layers = torch.nn.ModuleList(list(model.encoder.layers[:6]))
model.config.num_hidden_layers = 6
model = model.to(DEVICE)
for parameter in model.parameters():
    parameter.requires_grad_(False)


Frozen HuBERT extraction on NVIDIA GeForce RTX 4060 Laptop GPU


In [6]:
def hubert_vector(wave):
    parts = []
    with torch.inference_mode():
        for chunk in audio_chunks(wave):
            values = extractor(chunk.numpy(), sampling_rate=SAMPLE_RATE,
                               return_tensors='pt')['input_values'].to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == 'cuda'):
                output = model(values)
            parts.append(output.last_hidden_state[0].float().cpu())
    return summarize_frames(parts)

hubert_cache = FEATURES / 'hubert_cache'
hubert_cache.mkdir(exist_ok=True)
for row in tqdm(manifest.itertuples(index=False), total=len(manifest), desc='HuBERT'):
    destination = hubert_cache / f'{row.utterance_id}.npy'
    if not destination.exists():
        save_vector(hubert_cache, row.utterance_id, hubert_vector(load_audio(row.path)))
print('HuBERT vectors cached:', len(list(hubert_cache.glob('*.npy'))))


HuBERT:   0%|          | 0/13310 [00:00<?, ?it/s]

HuBERT:  28%|██▊       | 3752/13310 [00:00<00:00, 37232.78it/s]

HuBERT:  58%|█████▊    | 7654/13310 [00:00<00:00, 38276.92it/s]

HuBERT:  88%|████████▊ | 11756/13310 [00:00<00:00, 39462.46it/s]

HuBERT: 100%|██████████| 13310/13310 [00:00<00:00, 38935.23it/s]

HuBERT vectors cached: 13310


## 4. Consolidate and verify caches

These files are the fixed inputs to the two classifier heads. An ID mismatch stops the experiment.


In [7]:
ids = np.asarray(manifest.utterance_id.astype(str).tolist(), dtype=str)
for name, width in [('mfcc',120), ('hubert',1536)]:
    cache_dir = FEATURES / f'{name}_cache'
    missing = [item for item in ids if not (cache_dir / f'{item}.npy').exists()]
    assert not missing, f'{name}: {len(missing)} missing vectors'
    vectors = np.stack([np.load(cache_dir / f'{item}.npy') for item in ids])
    assert vectors.shape == (len(ids), width)
    assert np.isfinite(vectors).all()
    np.savez_compressed(FEATURES / f'{name}.npz', utterance_id=ids, X=vectors)
    print(name, vectors.shape, (FEATURES / f'{name}.npz').stat().st_size / 1e6, 'MB')
metadata = {'sample_rate':SAMPLE_RATE, 'chunk_seconds':10,
            'mfcc':{'coefficients':20,'deltas':2,'n_fft':400,'hop_length':160,'n_mels':40},
            'hubert':{'model':MODEL_ID,'revision':MODEL_REVISION,'layer':6,'pooling':'frame mean and standard deviation'},
            'torch':torch.__version__, 'torchaudio':torchaudio.__version__,
            'extractor_device':str(DEVICE), 'gpu':torch.cuda.get_device_name(DEVICE) if DEVICE.type == 'cuda' else None}
(FEATURES / 'feature_config.json').write_text(json.dumps(metadata, indent=2))


mfcc (13310, 120) 6.094923 MB


hubert (13310, 1536) 75.683206 MB


490